In [ ]:
# Hugging Face login
from huggingface_hub import login

login()

In [ ]:
# Load the MedGemma tokenizer
from transformers import AutoTokenizer

MODEL_ID = "google/medgemma-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded successfully")
print("Tokenizer class:", type(tokenizer).__name__)
print("Model max length:", tokenizer.model_max_length)

In [ ]:
from pathlib import Path

CHUNKS_FILE = Path("all_patient_chunks.jsonl")

In [ ]:
# Tokenising all the patient context files 

from pathlib import Path
import pandas as pd
import re

# Change this if patient_contexts is not in the notebook's current directory
PATIENT_DIR = Path("patient_contexts")

results = []

# Expected format: patient_subject-id_hadm-id.txt
pattern = re.compile(r"patient_(\d+)_(\d+)\.txt$")

for file_path in sorted(PATIENT_DIR.glob("patient_*.txt")):
    
    match = pattern.match(file_path.name)
    
    if not match:
        print(f"Skipping unexpected filename: {file_path.name}")
        continue
    
    subject_id = match.group(1)
    hadm_id = match.group(2)
    
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    # We only need the number of tokens
    token_count = len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )
    
    results.append({
        "subject_id": subject_id,
        "hadm_id": hadm_id,
        "file_name": file_path.name,
        "characters": len(text),
        "words": len(text.split()),
        "tokens": token_count
    })

df_tokens = pd.DataFrame(results)

print("Number of patient cases:", len(df_tokens))

display(df_tokens.head())

In [ ]:
# Statistical analysis of the number of token distribution
# Descriptive statistics of whole-case token counts
print("\nToken distribution:")

print(
    df_tokens["tokens"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nSmallest case:")
display(df_tokens.nsmallest(1, "tokens"))

print("\nLargest case:")
display(df_tokens.nlargest(1, "tokens"))

In [ ]:
# Model 1 final prediction prompt
MODEL1_PROMPT = """
You are given the available clinical record for a single hospital admission.

Using only the information provided in the patient record, predict the patient's
status at the end of the admission.

Your prediction must contain the following two outputs:

1. DISCHARGE DIAGNOSES

Primary diagnosis:
- Predict the most likely primary discharge diagnosis.

Secondary diagnoses:
- Predict the likely secondary discharge diagnoses.

2. DISCHARGE MANAGEMENT

Disposition:
- Predict the likely discharge disposition.

Discharge medications:
- Predict the likely medications at discharge.

Follow-up:
- Predict the likely follow-up requirements.

Ongoing care or monitoring:
- Predict the likely ongoing care or monitoring requirements.

Patient instructions/recommendations:
- Predict the likely discharge instructions or recommendations.

Use only information supported by the provided patient record.
Do not invent unsupported clinical information.
If a management component cannot reasonably be predicted from the available
information, state "Not specified."

If the predicted disposition is "Deceased", state "Not specified" for:
- Discharge medications
- Follow-up
- Ongoing care or monitoring
- Patient instructions/recommendations

PATIENT RECORD:
"""

In [ ]:
# Because MedGemma uses a chat template, the raw string will be tokenized as an actual user message rather than being tokenized as just a raw string. 
# Measure Model 1 prompt + chat-template overhead
messages = [
    {
        "role": "user",
        "content": MODEL1_PROMPT
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

prompt_tokens = tokenizer.encode(
    formatted_prompt,
    add_special_tokens=False
)

print("Raw prompt characters:", len(MODEL1_PROMPT))
print("Prompt + chat-template tokens:", len(prompt_tokens))

In [ ]:
# Clear unused GPU memory
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Evidence-extraction prompt
EVIDENCE_EXTRACTION_PROMPT = """
You are given one part of a patient's clinical record.

Extract and compress ONLY the clinical facts explicitly present in this record
part. This output will be used as evidence by another model call later.

DO NOT:
- Predict or state discharge diagnoses.
- Predict discharge disposition.
- Recommend or predict discharge medications.
- Recommend or predict follow-up.
- Generate patient instructions.
- Make clinical recommendations.
- Add information that is not explicitly supported by the supplied text.
- State that information is absent, unavailable, or not listed simply
because it does not appear in this record part. Omit categories that are not
represented in the supplied record part.
- Create headings or statements for information that is not present.
- Infer a trend unless the supplied record explicitly documents the
  trend or the chronological measurements clearly demonstrate it.


Preserve clinically relevant:
- symptoms and history
- examination findings
- laboratory and microbiology findings
- important trends over time
- imaging and diagnostic findings
- medications and treatments actually documented
- procedures and interventions actually documented
- changes in clinical condition
- response to treatment
- supportive-care requirements
- functional or care needs explicitly documented
- important negative findings
- relevant temporal information
- For repeated measurements, do not reproduce every value. Preserve clinically
  important abnormalities and meaningful changes over time, including
  representative values when useful.


Remove duplicate and redundant information.
Do not repeat the same fact more than once.

Output only a concise structured list of the clinical evidence present in the
supplied record. STRICTLY OMIT any category that is not represented in the supplied record.
Do not write "not listed", "not available", "none", or similar placeholders.

CLINICAL RECORD PART:
"""

In [ ]:
# Load MedGemma model and processor
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText


processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("MedGemma loaded successfully")
print("Model device:", model.device)

In [ ]:
# Define the operational input limit and measure the actual formatted MedGemma input
# Operational input limit + complete-input token counter
OPERATIONAL_INPUT_LIMIT = 20_000


def count_model_input_tokens(record_text, prompt, processor):
    """
    Count the tokens in the complete formatted MedGemma input,
    including the instruction prompt, patient record, and
    chat-template overhead.
    """

    full_prompt = prompt + "\n" + record_text

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": full_prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    return inputs["input_ids"].shape[-1]

In [ ]:
# this function sends one patient batch to MedGemma using EVIDENCE_EXTRACTION_PROMPT and returns the compressed clinical evidence
def extract_batch_evidence(
    batch,
    model,
    processor,
    max_new_tokens=1024
):
    batch_input = f"""
SECTION: {batch['section']}
SECTION PART: {batch['section_part']}

{batch['text']}
"""

    full_prompt = (
        EVIDENCE_EXTRACTION_PROMPT
        + "\n"
        + batch_input
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": full_prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4
        )

    generated_tokens = outputs[0][input_length:]

    evidence = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    generated_token_count = len(generated_tokens)

    hit_generation_limit = (
        generated_token_count >= max_new_tokens
    )

    return {
        "evidence": evidence,
        "generated_token_count": generated_token_count,
        "hit_generation_limit": hit_generation_limit
    }

In [ ]:
# Count the complete formatted input for an evidence-extraction call
def count_evidence_input_tokens(record_text, processor):
    """
    Count the complete formatted MedGemma input for an
    evidence-extraction call.
    """

    full_prompt = (
        EVIDENCE_EXTRACTION_PROMPT
        + "\n"
        + record_text
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": full_prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    return inputs["input_ids"].shape[-1]

In [ ]:
# step1- Build a reusable section-level representation
from collections import defaultdict
import json


def load_patient_sections(
    subject_id,
    hadm_id,
    tokenizer,
    chunks_file=CHUNKS_FILE
):
    """
    Reconstruct a patient's record at section level from the
    existing patient chunks.

    Returns one dictionary per section containing:
    - section name
    - ordered chunk IDs
    - ordered original chunks
    - reconstructed section text
    - section token count
    """

    sections = defaultdict(list)

    with open(chunks_file, "r", encoding="utf-8") as f:
        for line in f:
            chunk = json.loads(line)

            if (
                str(chunk["subject_id"]) == str(subject_id)
                and str(chunk["hadm_id"]) == str(hadm_id)
            ):
                sections[chunk["section"]].append(chunk)

    if not sections:
        raise ValueError(
            f"No chunks found for subject_id={subject_id}, "
            f"hadm_id={hadm_id}"
        )

    patient_sections = []

    for section, section_chunks in sections.items():

        # Preserve original chunk order
        section_chunks.sort(
            key=lambda x: x.get("chunk_id", 0)
        )

        section_text = "\n\n".join(
            chunk["text"].strip()
            for chunk in section_chunks
            if chunk["text"].strip()
        )

        token_count = len(
            tokenizer.encode(
                section_text,
                add_special_tokens=False
            )
        )

        patient_sections.append({
            "section": section,
            "chunks": section_chunks,
            "chunk_ids": [
                chunk.get("chunk_id")
                for chunk in section_chunks
            ],
            "text": section_text,
            "token_count": token_count,
            "compressed": False
        })

    return patient_sections

In [ ]:
# Step 2 — Reconstruct the current case and apply the real gate
def build_patient_representation(patient_sections):
    """
    Build the current complete patient representation
    from the section-level structure.

    Uses each section's current text, whether original
    or later replaced by compressed evidence.
    """

    parts = []

    for section in patient_sections:
        section_text = section["text"].strip()

        if not section_text:
            continue

        parts.append(
            f"{section['section']}\n"
            f"{section_text}"
        )

    return "\n\n".join(parts)

In [ ]:
# Step 3 — Select the largest uncompressed section

def get_largest_uncompressed_section(patient_sections):
    """
    Return the largest section that has not yet been compressed.
    """

    candidates = [
        section
        for section in patient_sections
        if not section["compressed"]
    ]

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda x: x["token_count"]
    )

In [ ]:
# Step 4 — Split the selected section only when necessary
def prepare_safe_section_batches(
    section,
    processor,
    operational_limit=OPERATIONAL_INPUT_LIMIT
):
    """
    Prepare safe evidence-extraction batches for one selected section.

    If the complete section fits within the operational input limit
    after adding the evidence-extraction prompt and chat template,
    return it as one batch.

    Otherwise, greedily combine its existing atomic chunks while
    ensuring every complete formatted evidence-extraction call
    remains within the operational limit.

    Original chunks are never split.
    """

    section_name = section["section"]
    section_text = section["text"].strip()

    # First check whether the whole section fits in one call
    whole_input_tokens = count_evidence_input_tokens(
        record_text=section_text,
        processor=processor
    )

    if whole_input_tokens <= operational_limit:
        return [{
            "section": section_name,
            "section_part": 1,
            "chunk_ids": section["chunk_ids"],
            "text": section_text,
            "input_token_count": whole_input_tokens
        }]

    # Otherwise split using the existing atomic chunks
    safe_batches = []

    current_chunks = []
    current_text = ""
    section_part = 1

    for chunk in section["chunks"]:

        chunk_text = chunk["text"].strip()

        if not chunk_text:
            continue

        if not current_chunks:
            candidate_text = chunk_text
        else:
            candidate_text = (
                current_text
                + "\n\n"
                + chunk_text
            )

        candidate_input_tokens = count_evidence_input_tokens(
            record_text=candidate_text,
            processor=processor
        )

        if candidate_input_tokens <= operational_limit:
            current_chunks.append(chunk)
            current_text = candidate_text

        else:
            # Save the current safe batch
            if current_chunks:
                current_input_tokens = count_evidence_input_tokens(
                    record_text=current_text,
                    processor=processor
                )

                safe_batches.append({
                    "section": section_name,
                    "section_part": section_part,
                    "chunk_ids": [
                        c.get("chunk_id")
                        for c in current_chunks
                    ],
                    "text": current_text,
                    "input_token_count": current_input_tokens
                })

                section_part += 1

            # Check whether the new chunk alone is safe
            chunk_input_tokens = count_evidence_input_tokens(
                record_text=chunk_text,
                processor=processor
            )

            if chunk_input_tokens > operational_limit:
                raise ValueError(
                    f"Chunk {chunk.get('chunk_id')} in section "
                    f"'{section_name}' requires "
                    f"{chunk_input_tokens:,} formatted input tokens, "
                    f"which exceeds the operational limit of "
                    f"{operational_limit:,}."
                )

            current_chunks = [chunk]
            current_text = chunk_text

    # Save final batch
    if current_chunks:
        current_input_tokens = count_evidence_input_tokens(
            record_text=current_text,
            processor=processor
        )

        safe_batches.append({
            "section": section_name,
            "section_part": section_part,
            "chunk_ids": [
                c.get("chunk_id")
                for c in current_chunks
            ],
            "text": current_text,
            "input_token_count": current_input_tokens
        })

    return safe_batches

In [ ]:
def split_truncated_batch(batch, section):
    """
    Split a truncated evidence batch into two smaller batches
    using the original atomic chunks.

    Original chunks are never split.
    """

    batch_chunk_ids = set(batch["chunk_ids"])

    batch_chunks = [
        chunk
        for chunk in section["chunks"]
        if chunk.get("chunk_id") in batch_chunk_ids
    ]

    if len(batch_chunks) <= 1:
        raise RuntimeError(
            f"Cannot further split truncated batch in "
            f"{section['section']} because it contains only one chunk."
        )

    midpoint = len(batch_chunks) // 2

    chunk_groups = [
        batch_chunks[:midpoint],
        batch_chunks[midpoint:]
    ]

    split_batches = []

    for group in chunk_groups:

        text = "\n\n".join(
            chunk["text"].strip()
            for chunk in group
            if chunk["text"].strip()
        )

        input_token_count = count_evidence_input_tokens(
            record_text=text,
            processor=processor
        )

        split_batches.append({
            "section": section["section"],
            "section_part": None,
            "chunk_ids": [
                chunk.get("chunk_id")
                for chunk in group
            ],
            "text": text,
            "input_token_count": input_token_count
        })

    return split_batches

In [ ]:
# Step 5 — Compress all batches of the selected section
def compress_section(
    section,
    model,
    processor,
    operational_limit=OPERATIONAL_INPUT_LIMIT,
    max_new_tokens=1024
):
    """
    Compress one selected section.

    Batches are first constructed to satisfy the input-context limit.

    If an evidence-generation call reaches max_new_tokens,
    that batch is split into smaller groups of its original
    atomic chunks and retried recursively.
    """

    initial_batches = prepare_safe_section_batches(
        section=section,
        processor=processor,
        operational_limit=operational_limit
    )

    evidence_parts = []
    batch_generation_info = []

    def process_batch(batch, label):

        print(
            f"Compressing {section['section']} "
            f"{label} "
            f"({batch['input_token_count']:,} input tokens)"
        )

        result = extract_batch_evidence(
            batch=batch,
            model=model,
            processor=processor,
            max_new_tokens=max_new_tokens
        )

        # Generation completed normally
        if not result["hit_generation_limit"]:

            if result["evidence"]:
                evidence_parts.append(result["evidence"])

            batch_generation_info.append({
                "batch": label,
                "chunk_ids": batch["chunk_ids"],
                "input_tokens": batch["input_token_count"],
                "generated_tokens": result["generated_token_count"],
                "hit_generation_limit": False
            })

            return

        # Generation was truncated
        print(
            f"WARNING: {section['section']} {label} "
            f"reached the {max_new_tokens}-token limit."
        )

        batch_generation_info.append({
            "batch": label,
            "chunk_ids": batch["chunk_ids"],
            "input_tokens": batch["input_token_count"],
            "generated_tokens": result["generated_token_count"],
            "hit_generation_limit": True
        })

        # Split truncated batch
        smaller_batches = split_truncated_batch(
            batch=batch,
            section=section
        )

        print(
            f"Re-splitting {label} into "
            f"{len(smaller_batches)} smaller batches."
        )

        # Process smaller batches in original order
        for j, smaller_batch in enumerate(
            smaller_batches,
            start=1
        ):
            process_batch(
                smaller_batch,
                label=f"{label}.{j}"
            )

    # Process initial safe batches in order
    for i, batch in enumerate(initial_batches, start=1):

        process_batch(
            batch,
            label=f"part {i}/{len(initial_batches)}"
        )

    compressed_text = "\n\n".join(evidence_parts)

    compressed_token_count = len(
        tokenizer.encode(
            compressed_text,
            add_special_tokens=False
        )
    )

    unresolved_truncation = any(
        item["hit_generation_limit"]
        for item in batch_generation_info
        if not any(
            str(other["batch"]).startswith(
                str(item["batch"]) + "."
            )
            for other in batch_generation_info
        )
    )

    return {
        "text": compressed_text,
        "token_count": compressed_token_count,
        "num_batches": len(initial_batches),
        "hit_generation_limit": unresolved_truncation,
        "batch_generation_info": batch_generation_info
    }

In [ ]:
def adapt_patient_to_context_limit(
    patient_sections,
    model,
    processor,
    tokenizer,
    final_prompt=MODEL1_PROMPT,
    operational_limit=OPERATIONAL_INPUT_LIMIT,
    max_new_tokens=1024
):
    """
    Greedily compress the largest remaining uncompressed section
    until the complete formatted final-model input fits within
    the operational context limit.

    Records compression and generation-limit information for
    each compressed section.
    """

    compression_log = []

    while True:

        # Rebuild current patient representation
        current_patient_text = build_patient_representation(
            patient_sections
        )

        # Check complete formatted final input
        current_input_tokens = count_model_input_tokens(
            record_text=current_patient_text,
            prompt=final_prompt,
            processor=processor
        )

        print(
            f"\nCurrent complete input: "
            f"{current_input_tokens:,} tokens"
        )

        # Stopping condition
        if current_input_tokens <= operational_limit:
            print("STOP: patient case fits the operational limit.")

            return {
                "patient_text": current_patient_text,
                "input_token_count": current_input_tokens,
                "compression_log": compression_log
            }

        # Select largest remaining uncompressed section
        selected_section = get_largest_uncompressed_section(
            patient_sections
        )

        if selected_section is None:
            raise RuntimeError(
                "The case still exceeds the operational limit, "
                "but no uncompressed sections remain."
            )

        original_token_count = selected_section["token_count"]

        print(
            f"Selected section: "
            f"{selected_section['section']} "
            f"({original_token_count:,} tokens)"
        )

        # Compress selected section
        compressed_result = compress_section(
            section=selected_section,
            model=model,
            processor=processor,
            operational_limit=operational_limit,
            max_new_tokens=max_new_tokens
        )

        # Replace original section with compressed evidence
        selected_section["text"] = compressed_result["text"]
        selected_section["token_count"] = (
            compressed_result["token_count"]
        )
        selected_section["compressed"] = True

        # Store full compression information
        compression_log.append({
            "section": selected_section["section"],
            "original_tokens": original_token_count,
            "compressed_tokens": compressed_result["token_count"],
            "evidence_batches": compressed_result["num_batches"],
            "hit_generation_limit": compressed_result[
                "hit_generation_limit"
            ],
            "batch_generation_info": compressed_result[
                "batch_generation_info"
            ]
        })

        print(
            f"{selected_section['section']}: "
            f"{original_token_count:,} -> "
            f"{compressed_result['token_count']:,} tokens"
        )

In [ ]:
# Select the longest patient case for stress-testing

largest_case = df_tokens.loc[
    df_tokens["tokens"].idxmax()
]

subject_id = largest_case["subject_id"]
hadm_id = largest_case["hadm_id"]

print("Testing longest patient case")
print("Subject ID:", subject_id)
print("HADM ID:", hadm_id)
print("Raw tokens:", f"{largest_case['tokens']:,}")

In [ ]:
patient_sections = load_patient_sections(
    subject_id=subject_id,
    hadm_id=hadm_id,
    tokenizer=tokenizer
)

reset_patient_text = build_patient_representation(
    patient_sections
)

reset_input_tokens = count_model_input_tokens(
    record_text=reset_patient_text,
    prompt=MODEL1_PROMPT,
    processor=processor
)

print("Compressed sections:", sum(
    section["compressed"]
    for section in patient_sections
))

print(
    f"Starting complete Model 1 input: "
    f"{reset_input_tokens:,} tokens"
)

In [ ]:
adaptive_result = adapt_patient_to_context_limit(
    patient_sections=patient_sections,
    model=model,
    processor=processor,
    tokenizer=tokenizer
)

In [ ]:
final_patient_text = adaptive_result["patient_text"]

print("=" * 80)
print("FINAL PATIENT REPRESENTATION SENT TO MODEL 1")
print("=" * 80)
print(final_patient_text)

In [ ]:
print("\nFINAL ADAPTIVE COMPRESSION SUMMARY\n")

print(
    f"Final complete Model 1 input: "
    f"{adaptive_result['input_token_count']:,} tokens"
)

print("\nCompressed sections:")

for item in adaptive_result["compression_log"]:

    print(
        f"\n{item['section']}: "
        f"{item['original_tokens']:,} -> "
        f"{item['compressed_tokens']:,} tokens | "
        f"unresolved truncation = "
        f"{item['hit_generation_limit']}"
    )

    for batch in item["batch_generation_info"]:
        print(
            f"    {batch['batch']}: "
            f"{batch['input_tokens']:,} input | "
            f"{batch['generated_tokens']:,} generated | "
            f"hit limit = {batch['hit_generation_limit']}"
        )

In [ ]:
def generate_model1_prediction(
    patient_text,
    model,
    processor,
    max_new_tokens=2048
):
    """
    Generate the final Model 1 prediction from the patient
    representation after context management.
    """

    full_prompt = (
        MODEL1_PROMPT
        + "\n"
        + patient_text
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": full_prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    print(
        f"Final Model 1 formatted input: "
        f"{input_length:,} tokens"
    )

    if input_length > OPERATIONAL_INPUT_LIMIT:
        raise ValueError(
            f"Final input contains {input_length:,} tokens, "
            f"which exceeds the operational limit of "
            f"{OPERATIONAL_INPUT_LIMIT:,}."
        )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4
    )
    

    generated_tokens = outputs[0][input_length:]

    prediction = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "prediction": prediction,
        "input_token_count": input_length,
        "generated_token_count": len(generated_tokens),
        "hit_generation_limit": (
            len(generated_tokens) >= max_new_tokens
        )
    }

In [ ]:
model1_result = generate_model1_prediction(
    patient_text=adaptive_result["patient_text"],
    model=model,
    processor=processor
)

print("\nMODEL 1 PREDICTION\n")
print(model1_result["prediction"])

print("\n--- Generation information ---")
print(
    "Input tokens:",
    f"{model1_result['input_token_count']:,}"
)
print(
    "Generated tokens:",
    f"{model1_result['generated_token_count']:,}"
)
print(
    "Hit generation limit:",
    model1_result["hit_generation_limit"]
)